# UIRAPURU — primeira execução (`00_uirapuru_first_run`)

Instrumentação da **simulação de primeira luz** do UIRAPURU com:

- **LO do DDC em 1120 MHz** (notch DC esperado) ± 125 MHz → 4096 canais;
- integração de **30 s**;
- constelações **GPS, GALILEO, GLONASS, BEIDOU** (satélites) + **fontes locais (Sol)**;
- padrão de feixe `beampattern_horn01.csv`, apontamento **fixo** alt=80°, az=9°;
- **EIRP dos satélites recalculado** em `signals.json` (deste diretório), derivado de
  `BINGO_ASSEMBLY/notebooks/04_GNSS_SIGNAL_ESTIMATION.ipynb`;
- **fluxo solar** pelo novo provider **NoRP/NRO** (XDR/IDL SAVE).

Arquivos do diretório:

| arquivo | papel |
|---|---|
| `signals.json` | EIRP por constelação **e** por satélite (novos valores) |
| `settings_first_run.toml` | perfil do telescópio + window + provider solar + signals |
| `run_first_run.py` | runner (estimativa + execução, daemonizável) |
| `output/` | resultados (`uirapuru_first_run.nc`, sidecar `.json`, `run.log`) |


## 0. Âncora de diretório

Torna os caminhos relativos independentes de onde o kernel começou.

In [ ]:
import os
from pathlib import Path

from radioskysims_core.paths import get_project_root

os.chdir(get_project_root())
HERE = Path("notebooks/00_uirapuru_first_run").resolve()
print("cwd →", os.getcwd())

## 1. Parâmetros do experimento

In [ ]:
CENTRE_FREQUENCY_MHZ = 1120.0   # LO do DDC; notch esperado
NUMBER_OF_CHANNELS = 4096
REQUESTED_INTEGRATION_TIME_S = 30.0
CONSTELLATIONS = ["GPS", "GALILEO", "GLONASS", "BEIDOU"]
INCLUDE_SATELLITES = True
INCLUDE_LOCAL_SOURCES = True
BEAM_PATTERN_PATH = Path("src/radioskysims/resources/beams/beampattern_horn01.csv")
POINTING_ALT_DEG = 80.0
POINTING_AZ_DEG = 9.0
SETTINGS_FILE = HERE / "settings_first_run.toml"
TELESCOPE = "uirapuru_first_run"
print("beam existe:", (get_project_root() / BEAM_PATTERN_PATH).exists())

## 2. Orquestrador a partir do settings da execução

`settings_first_run.toml` aponta `[signals] sat_signals` para o `signals.json`
deste diretório e `[solar_flux] source = "nro"`.

In [ ]:
from radioskysims_simulation import SimOrchestrator

orch = SimOrchestrator.from_settings(telescope=TELESCOPE, settings_file=str(SETTINGS_FILE))
ctx = orch._ctx
fr = ctx.frequency_range
print("FR (MHz):", fr.min, fr.max, fr.resolution)
print("canais:", len(fr.value))
print("constelações padrão:", orch.default_constellations())

## 3. Estimativa de tempo

Extrapolação linear a partir de execuções curtas (3 e 6 amostras).

In [ ]:
import sys
sys.path.insert(0, str(HERE))
import run_first_run as R

full = orch.default_time_range()
print("instantes:", full.to_datetime64().shape[0])
estimate = R.estimate_runtime(orch, full)
estimate

## 4. Execução

> Para a execução longa **daemonizada** (sobrevive ao fechamento do console):
> ```bash
> cd <raiz do projeto>
> setsid nohup .venv/bin/python notebooks/00_uirapuru_first_run/run_first_run.py \
>     > notebooks/00_uirapuru_first_run/output/run.log 2>&1 &
> ```

Use `use_processor=True` para streaming automático a disco quando a saída
excede a memória.

In [ ]:
import time

t0 = time.monotonic()
result = orch.run(
    constellations=CONSTELLATIONS,
    time_range=full,
    include_satellites=INCLUDE_SATELLITES,
    include_local_sources=INCLUDE_LOCAL_SOURCES,
    include_radio_sources=False,
    use_processor=True,
)
print(f"run() em {time.monotonic() - t0:.1f}s")

ds = result.simulation_result.dataset
print("dims:", dict(ds.sizes))
print(result.format_summary()[:1200])

## 5. Persistência

In [ ]:
out = HERE / "output" / "uirapuru_first_run.nc"
out.parent.mkdir(parents=True, exist_ok=True)
sidecar = result.save_result(str(out))
print("dataset:", out)
print("sidecar:", sidecar)